In [ ]:
%%capture
!pip install llama-index llama-index-embeddings-openai qdrant-client llama-index-vector-stores-qdrant llama-index llama-index-llms-openai llama-index-vector-stores-faiss faiss-cpu

In [ ]:
import os
from getpass import getpass
import nest_asyncio
from dotenv import load_dotenv
from llama_index.llms.openai import OpenAI
from llama_index.core.settings import Settings

nest_asyncio.apply()

load_dotenv()

In [ ]:
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY') or getpass("Enter your OpenAI API key: ")
TAVILY_API_KEY=os.environ.get('TAVILY_API_KEY') or getpass("Enter your TAVILY API key: ")

In [ ]:
from llama_index.embeddings.openai import OpenAIEmbedding


Settings.llm = OpenAI(model="gpt-4o-mini-2024-07-18", api_key=OPENAI_API_KEY)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

In [ ]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.extractors import TitleExtractor, SummaryExtractor
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import MetadataMode


def build_pipeline():
    transformations = [
        SentenceSplitter(chunk_size=250, chunk_overlap=20),
        TitleExtractor(
            llm=Settings.llm, metadata_mode=MetadataMode.EMBED, num_workers=8
        ),
        SummaryExtractor(
            llm=Settings.llm, metadata_mode=MetadataMode.EMBED, num_workers=8
        ),
        Settings.embed_model
    ]

    return IngestionPipeline(transformations=transformations)

In [ ]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(r"C:\Users\anteb\PycharmProjects\JupyterProject\naive_rag\data\ai_articles").load_data()

In [ ]:
documents[0].metadata

In [ ]:
# from naive_rag.helpers.IngestionCacheManager import SmartIngestionCache
# from llama_index.core.ingestion import IngestionCache, IngestionPipeline
#
# ingest_cache = SmartIngestionCache().get_cache()
#
# pipeline = build_pipeline()
#
# nodes = pipeline.run(documents = documents)

In [ ]:

import faiss
from llama_index.core.extractors import TitleExtractor, SummaryExtractor
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import MetadataMode
from naive_rag.helpers.IngestionCacheManager import SmartIngestionCache
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.storage.docstore import SimpleDocumentStore


# Create a FAISS vector store instance.
# Note: Set the embedding dimension according to the output size of your embedding model.
# For example, if "text-embedding-3-small" returns 384-dim vectors, use embedding_dim=384.
from llama_index.vector_stores.faiss import FaissVectorStore


# create a faiss index
d = 1536  # dimension
faiss_index = faiss.IndexFlatL2(d)

vector_store = FaissVectorStore(faiss_index=faiss_index)

ingest_cache = SmartIngestionCache().get_cache()

# Build the ingestion pipeline, now passing in the FAISS vector store.
pipeline = IngestionPipeline(
    transformations=build_pipeline().transformations,
    docstore=SimpleDocumentStore(),
    vector_store=vector_store,
    cache=ingest_cache,
)

# Process your documents: this will split, extract titles/summaries, embed the nodes,
# and store them in the FAISS index.
nodes = pipeline.run(documents=documents)

In [ ]:
print("Total number of vectors in FAISS index:", faiss_index.ntotal)
print("Number of nodes:", len(nodes))
print("Number of documents:", len(documents))

In [ ]:
from llama_index.core import VectorStoreIndex

# Create an index using llama_index.
# This index is built from your preprocessed nodes and the FAISS vector store.
index = VectorStoreIndex(nodes, vector_store=vector_store)

# Build a retriever from the index.
# The retriever is configured to return the top 3 most similar nodes for any given query.
retriever = index.as_retriever(search_kwargs={"k": 3})

# Define an async tool function to query the FAISS database.
# This tool can be added to your agent's list of tools for retrieval tasks.
import asyncio




In [ ]:
print(nodes[6].metadata)

In [ ]:
%pip install tavily-python

In [ ]:
from tavily import AsyncTavilyClient
from llama_index.core.workflow import Context


async def search_web(query: str) -> str:
    """Useful for using the web to answer questions."""
    client = AsyncTavilyClient(api_key=TAVILY_API_KEY)
    return str(await client.search(query))

import asyncio

async def search_faiss(query: str) -> str:
    """
    Searches the FAISS index for the given query and returns a compact response.
    """
    # Create a query engine from the index. In the newer API, use as_query_engine().
    query_engine = index.as_query_engine(response_mode="compact")
    # Run the query in a thread-safe manner.
    result = await asyncio.to_thread(query_engine.query, query)
    return str(result)



async def record_notes(ctx: Context, notes: str, notes_title: str) -> str:
    """Useful for recording notes on a given topic. Your input should be notes with a title to save the notes under."""
    current_state = await ctx.get("state")
    if "research_notes" not in current_state:
        current_state["research_notes"] = {}
    current_state["research_notes"][notes_title] = notes
    await ctx.set("state", current_state)
    return "Notes recorded."


async def write_report(ctx: Context, report_content: str) -> str:
    """Useful for writing a report on a given topic. Your input should be a markdown formatted report."""
    current_state = await ctx.get("state")
    current_state["report_content"] = report_content
    await ctx.set("state", current_state)
    return "Report written."


async def review_report(ctx: Context, review: str) -> str:
    """Useful for reviewing a report and providing feedback. Your input should be a review of the report."""
    current_state = await ctx.get("state")
    current_state["review"] = review
    await ctx.set("state", current_state)
    return "Report reviewed."

In [ ]:
import asyncio
from llama_index.core.agent.workflow import FunctionAgent, AgentWorkflow, AgentOutput, ToolCall, ToolCallResult, AgentStream

# Create a test agent that uses only the FAISS search tool.
test_agent = FunctionAgent(
    name="TestAgent",
    description="Agent for testing the FAISS search tool. When given a query, use the FAISS tool to retrieve information from the index.",
    system_prompt=(
        "You are TestAgent. When given a query, use the FAISS search tool to look up relevant information from the index."
    ),
    llm=Settings.llm,
    tools=[search_faiss],
)

# Set up an agent workflow with just this one test agent.
test_workflow = AgentWorkflow(
    agents=[test_agent],
    root_agent=test_agent.name,
    initial_state={},
)

# Define an async function to trigger the agent and print out events.
async def test_faiss_tool():
    handler = test_workflow.run(
        user_msg="Search the FAISS index for best description of RAG."
    )

    async for event in handler.stream_events():
        if hasattr(event, "current_agent_name"):
            print(f"\n{'='*50}\nAgent: {event.current_agent_name}\n{'='*50}\n")
        if isinstance(event, AgentOutput) and event.response.content:
            print("Output:", event.response.content)
        if isinstance(event, ToolCall):
            print("Tool Call:", event.tool_name, event.tool_kwargs)
        if isinstance(event, ToolCallResult):
            print("Tool Call Result:", event.tool_name, event.tool_output)

# Run the test.
await test_faiss_tool()


In [ ]:
from llama_index.core.agent.workflow import FunctionAgent, ReActAgent

research_agent = FunctionAgent(
    name="ResearchAgent",
    description="Useful for searching the web for information on a given topic and recording notes on the topic.",
    system_prompt=(
        "You are the ResearchAgent that can search the web for information on a given topic and record notes on the topic. "
        "Once notes are recorded and you are satisfied, you should hand off control to the WriteAgent to write a report on the topic. "
        "You should have at least some notes on a topic before handing off control to the WriteAgent."
    ),
    llm=Settings.llm,
    tools=[search_web, record_notes],
    can_handoff_to=["WriteAgent"],
)

write_agent = FunctionAgent(
    name="WriteAgent",
    description="Useful for writing a report on a given topic.",
    system_prompt=(
        "You are the WriteAgent that can write a report on a given topic. "
        "Your report should be in a markdown format. The content should be grounded in the research notes. "
        "Once the report is written, you should get feedback at least once from the ReviewAgent."
    ),
    llm=Settings.llm,
    tools=[write_report],
    can_handoff_to=["ReviewAgent", "ResearchAgent"],
)

review_agent = FunctionAgent(
    name="ReviewAgent",
    description="Useful for reviewing a report and providing feedback.",
    system_prompt=(
        "You are the ReviewAgent that can review the write report and provide feedback. "
        "Your review should either approve the current report or request changes for the WriteAgent to implement. "
        "If you have feedback that requires changes, you should hand off control to the WriteAgent to implement the changes after submitting the review."
    ),
    llm=Settings.llm,
    tools=[review_report],
    can_handoff_to=["WriteAgent"],
)

In [ ]:
from llama_index.core.agent.workflow import AgentWorkflow

agent_workflow = AgentWorkflow(
    agents=[research_agent, write_agent, review_agent],
    root_agent=research_agent.name,
    initial_state={
        "research_notes": {},
        "report_content": "Not written yet.",
        "review": "Review required.",
    },
)

In [ ]:
from llama_index.core.agent.workflow import (
    AgentInput,
    AgentOutput,
    ToolCall,
    ToolCallResult,
    AgentStream,
)

handler = agent_workflow.run(
    user_msg=(
        "Write me a report on computer system validation. "
        "Briefly describe the main steps and challenges"
        "Give recommendations on implementation of CSV"
    )
)

current_agent = None
current_tool_calls = ""
async for event in handler.stream_events():
    if (
        hasattr(event, "current_agent_name")
        and event.current_agent_name != current_agent
    ):
        current_agent = event.current_agent_name
        print(f"\n{'='*50}")
        print(f"🤖 Agent: {current_agent}")
        print(f"{'='*50}\n")

    # if isinstance(event, AgentStream):
    #     if event.delta:
    #         print(event.delta, end="", flush=True)
    # elif isinstance(event, AgentInput):
    #     print("📥 Input:", event.input)
    elif isinstance(event, AgentOutput):
        if event.response.content:
            print("📤 Output:", event.response.content)
        if event.tool_calls:
            print(
                "🛠️  Planning to use tools:",
                [call.tool_name for call in event.tool_calls],
            )
    elif isinstance(event, ToolCallResult):
        print(f"🔧 Tool Result ({event.tool_name}):")
        print(f"  Arguments: {event.tool_kwargs}")
        print(f"  Output: {event.tool_output}")
    elif isinstance(event, ToolCall):
        print(f"🔨 Calling Tool: {event.tool_name}")
        print(f"  With arguments: {event.tool_kwargs}")

In [ ]:
state = await handler.ctx.get("state")
print(state["report_content"])